# 실험3 결과 — act+TE / bimamba+TE / bimamba+carry+TE **K별 비교표**

두 학습/eval 노트북(`exp3_carry_te_ksweep`, `exp3_act_te_ksweep`)이 끝난 뒤 실행.
**읽기 전용** — 결과를 K별 표(SR + 떨림)로 정리하고 **이미지·CSV·zip**으로 저장(은지님께 바로 전달).

- 세 모델 전부 **같은 K·같은 100ep·TE 0.01·같은 seed** → 공정 비교.
- **Δcarry** = (bimamba+carry+TE) − (bimamba+TE): carry 전달이 TE 위에서 주는 이득(교수님 결정 지표).
- `K_LIST`·`SEEDS` 는 실험 노트북들과 **동일하게** 둘 것. 아직 안 나온 칸은 빈칸.


## 0) 부팅 + 헬퍼


In [ ]:
import sys, json, csv, time
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)
v23 = cf.v23
import numpy as np, smooth_metrics_paper as smp
importlib.reload(smp)

# ── 무엇을 보여주나 ────────────────────────────────────────────────────────
# K별로 act+TE / bimamba+TE / bimamba+carry+TE 를 같은 100ep 로 비교(SR + 떨림) → 표+이미지+CSV+zip.
#   두 학습/eval 노트북(exp3_carry_te_ksweep, exp3_act_te_ksweep)이 만든 결과를 읽기만 함.
TASK   = 'libero_10'
K_LIST = [10, 15, 20, 50, 100, 150]   # 두 실험 노트북과 동일하게
SEEDS  = [0]
N_EP   = 100
FS, STRIDE = cf.fps_of(TASK), 100
MIN_VALID = 10 * N_EP // 2            # overall n_ep >= 500 만 유효

# 3자 태그: act+TE = ak{K}_te / bimamba+TE = bk{K}_nocarry_te / bimamba+carry+TE = bk{K}_carry_te
VARIANTS = [('act+TE',           lambda K: f'ak{K}_te'),
            ('bimamba+TE',       lambda K: f'bk{K}_nocarry_te'),
            ('bimamba+carry+TE', lambda K: f'bk{K}_carry_te')]

OUT = cf.OUTPUT_BASE / 'share' / 'exp3_results'
OUT.mkdir(parents=True, exist_ok=True)
STAMP = time.strftime('%Y%m%d_%H%M')

def rec(tag, s):
    d = cf.OUTPUT_BASE / 'eval_clean' / TASK / v23.MODEL_DIR_NAMES.get(tag, tag) / f'seed{s}'
    if not d.is_dir(): return None
    best = None
    for info in d.rglob('eval_info.json'):
        try: ov = json.loads(info.read_text()).get('overall', {})
        except Exception: continue
        n = ov.get('n_ep', ov.get('n_episodes')) or 0
        if best is None or n > best['n']:
            best = {'sr': ov.get('pc_success'), 'n': n, 'act': (info.parent/'actions').is_dir(), 'p': info.parent}
    return best
def sr_stats(tag):
    vals = [rec(tag, s)['sr'] for s in SEEDS
            if rec(tag, s) and (rec(tag, s)['n'] or 0) >= MIN_VALID and rec(tag, s)['sr'] is not None]
    if not vals: return None, None, 0
    return float(np.mean(vals)), (float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0), len(vals)
def sm_of(tag):
    trajs = []
    for s in SEEDS:
        e = rec(tag, s)
        if e and (e['n'] or 0) >= MIN_VALID and e['act']:
            t = v23._load_action_trajs(e['p']/'actions') or []
            if len(t) >= 80: trajs += t
    return smp.aggregate_paper(trajs, boundary_stride=STRIDE, fs=FS) if trajs else None

def save_table_png(col_labels, cell_text, title, path):
    import matplotlib; matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    nrow, ncol = len(cell_text), len(col_labels)
    fig, ax = plt.subplots(figsize=(1.4 + 1.3 * ncol, 0.6 + 0.42 * (nrow + 1)))
    ax.axis('off')
    tb = ax.table(cellText=cell_text, colLabels=col_labels, loc='center', cellLoc='center')
    tb.auto_set_font_size(False); tb.set_fontsize(10)
    tb.auto_set_column_width(col=list(range(ncol))); tb.scale(1, 1.5)
    for (r, c), cell in tb.get_celld().items():
        if r == 0: cell.set_text_props(weight='bold'); cell.set_facecolor('#e8e8e8')
    ax.set_title(title, fontsize=12, pad=10)
    fig.savefig(path, dpi=150, bbox_inches='tight'); plt.close(fig)
    print('  이미지:', path)

print('eval:', cf.OUTPUT_BASE / 'eval_clean' / TASK, '| 저장:', OUT, '| 스냅샷', STAMP)

## 1) 상태 (어떤 K×변형이 채워졌나)


In [ ]:
# 어떤 (K × 변형)이 채워졌나 (유효 500+ep eval)
print(f'{"K":>5}' + ''.join(f'{name:>20}' for name, _ in VARIANTS))
for K in K_LIST:
    marks = []
    for name, fn in VARIANTS:
        m, sd, ns = sr_stats(fn(K))
        marks.append(f'{ns}/{len(SEEDS)} 완료' if ns else '—')
    print(f'{K:>5}' + ''.join(f'{x:>20}' for x in marks))

## 2) SR 표 (K × 3변형 + Δcarry)


In [ ]:
# ── SR 표 (K × 3변형, mean±std) + carry 이득 Δ(carry+TE − TE). CSV+PNG ──
rows = []
print(f'{"K":>5}' + ''.join(f'{n:>20}' for n, _ in VARIANTS) + f'{"Δcarry":>10}')
for K in K_LIST:
    cells, srv = [], {}
    for name, fn in VARIANTS:
        m, sd, ns = sr_stats(fn(K)); srv[name] = m
        cells.append(f'{m:.1f}±{sd:.1f}' if m is not None else '-')
    d = (srv['bimamba+carry+TE'] - srv['bimamba+TE'])
    dstr = f'{d:+.1f}' if (srv['bimamba+carry+TE'] is not None and srv['bimamba+TE'] is not None) else '-'
    print(f'{K:>5}' + ''.join(f'{c:>20}' for c in cells) + f'{dstr:>10}')
    rows.append({'K': K, **{n: (round(srv[n], 2) if srv[n] is not None else None) for n, _ in VARIANTS},
                 'delta_carry_te': (round(d, 2) if dstr != '-' else None)})
cols = ['K'] + [n for n, _ in VARIANTS] + ['delta_carry_te']
with open(OUT / f'sr_ksweep_{STAMP}.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows(rows)
img_col = ['K'] + [n for n, _ in VARIANTS] + ['Δcarry']
img_txt = [[str(r['K'])] + [('' if r[n] is None else f"{r[n]:.1f}") for n, _ in VARIANTS]
           + [('' if r['delta_carry_te'] is None else f"{r['delta_carry_te']:+.1f}")] for r in rows]
save_table_png(img_col, img_txt, f'LIBERO-10 SR vs K ({N_EP}ep/task)  {STAMP}', OUT / f'sr_ksweep_{STAMP}.png')
print('saved:', OUT / f'sr_ksweep_{STAMP}.csv')

## 3) 떨림 표 (K × 변형)


In [ ]:
# ── 떨림 표 (K × 변형): jerk / bnd / int / B·I / SPARC / sflip. CSV+PNG ──
rows = []
print(f'{"K":>5}  {"variant":<18}{"jerk":>9}{"bnd":>9}{"int":>9}{"B/I":>7}{"SPARC":>9}{"sflip":>9}{"n":>6}')
print('  방향:                        ↓        ↓        ↓     →1     →0        ↓')
for K in K_LIST:
    for name, fn in VARIANTS:
        a = sm_of(fn(K))
        if a:
            print(f'{K:>5}  {name:<18}{a["jerk_rms_mean"]:>9.4f}{a["boundary_jerk_rms_mean"]:>9.4f}'
                  f'{a["interior_jerk_rms_mean"]:>9.4f}{a["boundary_interior_ratio_mean"]:>7.2f}'
                  f'{a["sparc_mean"]:>9.2f}{a["sign_flip_rate_mean"]:>9.4f}{a["n_traj"]:>6}')
            rows.append({'K': K, 'variant': name, 'jerk_rms': round(a['jerk_rms_mean'], 5),
                         'boundary_jerk_rms': round(a['boundary_jerk_rms_mean'], 5),
                         'interior_jerk_rms': round(a['interior_jerk_rms_mean'], 5),
                         'boundary_interior_ratio': round(a['boundary_interior_ratio_mean'], 4),
                         'sparc': round(a['sparc_mean'], 3),
                         'sign_flip_rate': round(a['sign_flip_rate_mean'], 5), 'n_traj': a['n_traj']})
        else:
            print(f'{K:>5}  {name:<18}' + ' '*49 + '(빈칸)')
            rows.append({'K': K, 'variant': name, 'jerk_rms': None, 'boundary_jerk_rms': None,
                         'interior_jerk_rms': None, 'boundary_interior_ratio': None, 'sparc': None,
                         'sign_flip_rate': None, 'n_traj': 0})
with open(OUT / f'smooth_ksweep_{STAMP}.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['K', 'variant', 'jerk_rms', 'boundary_jerk_rms',
        'interior_jerk_rms', 'boundary_interior_ratio', 'sparc', 'sign_flip_rate', 'n_traj'])
    w.writeheader(); w.writerows(rows)
def _f(v, d): return '' if v is None else f'{v:.{d}f}'
img_col = ['K', 'variant', 'jerk↓', 'bnd↓', 'int↓', 'B/I→1', 'SPARC→0', 'sflip↓', 'n']
img_txt = [[str(r['K']), r['variant'], _f(r['jerk_rms'], 4), _f(r['boundary_jerk_rms'], 4),
            _f(r['interior_jerk_rms'], 4), _f(r['boundary_interior_ratio'], 2), _f(r['sparc'], 2),
            _f(r['sign_flip_rate'], 4), (str(r['n_traj']) if r['n_traj'] else '')] for r in rows]
save_table_png(img_col, img_txt, f'LIBERO-10 smoothness vs K (aloha, fs={FS})  {STAMP}', OUT / f'smooth_ksweep_{STAMP}.png')
print('saved:', OUT / f'smooth_ksweep_{STAMP}.csv')

## 4) 요약 + zip


In [ ]:
# ── 요약 + zip (표 CSV·PNG 는 위에서 저장됨) ──
import shutil
(OUT / f'README_{STAMP}.md').write_text(
    f'# 실험3 K-sweep 결과 (LIBERO-10, {N_EP}ep/task, {STAMP})\n\n'
    f'- 비교: act+TE / bimamba+TE / bimamba+carry+TE — 전부 같은 K·같은 {N_EP}ep·TE 0.01·seed {SEEDS}\n'
    f'- K: {K_LIST}\n- 떨림 = aloha 동일 스크립트, fs={FS}, 경계 stride={STRIDE}\n'
    f'- Δcarry = (bimamba+carry+TE) − (bimamba+TE): carry 전달이 TE 위에서 주는 SR 이득\n', encoding='utf-8')
zip_path = shutil.make_archive(str(cf.OUTPUT_BASE / 'share' / f'exp3_results_{STAMP}'), 'zip', root_dir=OUT)
print('보낼 파일:', zip_path)
for p in sorted(OUT.rglob('*')):
    if p.is_file(): print(f'  {p.relative_to(OUT)}  ({p.stat().st_size/1024:.0f} KB)')